# CDFI DiD Estimation Pipeline v2.0

Callaway & Sant'Anna (2021) Difference-in-Differences with Neural Network Nuisance Estimation

**Key improvements in v2.0:**
- Wide unit-level data structure
- Base-period grouped covariate projections (~19 instead of ~342)
- Single forward pass produces all (g,t) predictions
- On-the-fly outcome differencing

---

## Cell 1: Setup

In [ ]:
import sys
import os

# Set project root
project_root = "/path/to/your/project"  # <-- CHANGE THIS
os.chdir(project_root)
sys.path.insert(0, os.path.join(project_root, "code/python"))

# Import v2.0 modules
from modules import (
    # Config
    create_config, set_seed, print_config, get_device,
    # Data
    load_panel_data, create_unit_data, create_gt_info,
    # Covariates
    create_covariate_info,
    # Cross-fitting
    run_cross_fitting, validate_cross_fitting,
    # ATT
    compute_all_att, print_att_summary,
    # Inference
    add_bootstrap_inference, test_parallel_trends, compute_simple_att,
    # Aggregation
    aggregate_all, print_aggregation_summary,
    # Visualization
    plot_event_study, save_event_study,
    # Utils
    Timer
)

import torch
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"MPS: {torch.backends.mps.is_available()}")
print("\nSetup complete.")

## Cell 2: Configuration

In [ ]:
# =============================================================================
# FULL CONFIGURATION - EVERY HYPERPARAMETER EXPOSED
# =============================================================================

config = create_config(
    # -------------------------------------------------------------------------
    # DATA STRUCTURE
    # -------------------------------------------------------------------------
    # Primary outcome variable (without y_ prefix)
    outcome="sfr_pc",
    
    # Panel identifiers
    id_var="id",                    # Unit identifier
    time_var="time",                # Time period variable
    group_var="group",              # Treatment cohort variable
    cluster_var="cluster_county",   # Clustering variable for bootstrap
    
    # Covariate prefixes (identifies which columns to use)
    time_invariant_prefix="X_",     # Time-invariant covariates
    time_varying_prefix="V_",       # Time-varying covariates
    
    # Treatment coding
    never_treated_code=0,           # Code for never-treated units
    
    # Analysis window
    analysis_start=1996,            # First analysis period
    analysis_end=2014,              # Last analysis period
    
    # -------------------------------------------------------------------------
    # NEURAL NETWORK ARCHITECTURE
    # -------------------------------------------------------------------------
    architecture={
        # Input processing
        'input_projection_dim': 128,        # Dimension of base-period projections
        
        # Shared encoder
        'shared_layers': [256, 128],        # Hidden layer dimensions (list)
        
        # Per-(g,t) heads
        'outcome_head_layers': [64],        # Outcome head hidden layers
        'propensity_head_layers': [64],     # Propensity head hidden layers
        
        # Activation and regularization
        'activation': 'relu',               # Options: 'relu', 'gelu', 'silu', 'tanh'
        'dropout': 0.2,                     # Dropout probability (0 to disable)
        'layer_norm': True                  # Use LayerNorm (True/False)
    },
    
    # -------------------------------------------------------------------------
    # TRAINING
    # -------------------------------------------------------------------------
    training={
        'epochs': 100,                      # Maximum training epochs
        'batch_size': 512,                  # Batch size
        'validation_split': 0.2,            # Fraction held out for validation
        
        # Early stopping
        'early_stopping_patience': 15,      # Epochs without improvement before stopping
        'early_stopping_min_delta': 1e-4,   # Minimum improvement to count as progress
        
        # Gradient clipping
        'gradient_clip_norm': 1.0           # Max gradient norm (0 to disable)
    },
    
    # -------------------------------------------------------------------------
    # OPTIMIZER
    # -------------------------------------------------------------------------
    optimizer={
        'name': 'adamw',                    # Options: 'adamw', 'adam', 'sgd'
        'lr': 0.001,                        # Learning rate
        'weight_decay': 0.01,               # L2 regularization
        'betas': (0.9, 0.999)               # Adam beta parameters
    },
    
    # -------------------------------------------------------------------------
    # LEARNING RATE SCHEDULER
    # -------------------------------------------------------------------------
    scheduler={
        'name': 'cosine',                   # Options: 'none', 'cosine', 'step', 'plateau'
        # Cosine annealing parameters
        'T_max': 100,                       # Period of cosine annealing (epochs)
        'eta_min': 1e-6,                    # Minimum learning rate
        # Step scheduler parameters (only used if name='step')
        'step_size': 30,                    # Decay LR every N epochs
        'gamma': 0.1,                       # Multiplicative factor of LR decay
        # Plateau scheduler parameters (only used if name='plateau')
        'plateau_factor': 0.1,              # Factor to reduce LR by
        'plateau_patience': 10,             # Epochs to wait before reducing LR
        'plateau_min_lr': 1e-6              # Lower bound on learning rate
    },
    
    # -------------------------------------------------------------------------
    # LOSS FUNCTION WEIGHTS
    # -------------------------------------------------------------------------
    loss={
        'outcome_weight': 1.0,              # Weight on outcome (MSE) loss
        'propensity_weight': 1.0            # Weight on propensity (BCE) loss
    },
    
    # -------------------------------------------------------------------------
    # CROSS-FITTING (DML)
    # -------------------------------------------------------------------------
    cross_fitting={
        'n_folds': 2,                       # Number of cross-fitting folds
        'stratify_by': 'cluster_county',    # Stratification variable
        'seed': 42                          # Random seed for fold assignment
    },
    
    # -------------------------------------------------------------------------
    # PROPENSITY SCORE
    # -------------------------------------------------------------------------
    propensity={
        'min_ps': 0.001,                    # Minimum propensity score (trimming)
        'max_ps': 0.999                     # Maximum propensity score (trimming)
    },
    
    # -------------------------------------------------------------------------
    # INFERENCE (BOOTSTRAP)
    # -------------------------------------------------------------------------
    inference={
        'n_bootstrap': 1000,                # Number of bootstrap replications
        'alpha': 0.05,                      # Significance level (1-alpha CI)
        'uniform_bands': True,              # Compute uniform confidence bands
        'multiplier_dist': 'normal',        # Options: 'normal', 'rademacher'
        'seed': 123                         # Random seed for bootstrap
    },
    
    # -------------------------------------------------------------------------
    # EVENT STUDY AGGREGATION
    # -------------------------------------------------------------------------
    event_study={
        'pre_periods': 10,                  # Number of pre-treatment periods
        'post_periods': 10,                 # Number of post-treatment periods
        'reference_period': -1,             # Reference period for normalization
        'weight_by_group_size': True        # Weight by group size (True/False)
    },
    
    # -------------------------------------------------------------------------
    # MONITORING & OUTPUT
    # -------------------------------------------------------------------------
    monitoring={
        'verbose': True,                    # Print training progress
        'print_every': 10                   # Print every N epochs
    },
    
    # -------------------------------------------------------------------------
    # COMPUTATIONAL
    # -------------------------------------------------------------------------
    device='auto',                          # Options: 'auto', 'cpu', 'cuda', 'mps'
    seed=42                                 # Global random seed
)

# Print configuration summary
print_config(config)

# Set seeds and get device
set_seed(config.seed)
device = get_device(config)
print(f"\nDevice: {device}")

## Cell 3: Load Data and Create Unit-Level Structure

In [ ]:
data_path = os.path.join(project_root, "data/analysis/final_analysis_dataset.csv")

# Sample size for testing (None for full data)
sample_n = None  # e.g., 10000 for testing

timer = Timer()

# Load panel data
panel_df = load_panel_data(data_path, config, sample_n=sample_n)

# Create wide unit-level structure
unit_data = create_unit_data(panel_df, config)

# Create (g,t) pair info
gt_info = create_gt_info(unit_data, config)

# Create covariate masks by base period
covariate_info = create_covariate_info(unit_data, gt_info, config)

print(f"\nData preparation complete in {timer.elapsed():.1f}s")
print(f"\nSummary:")
print(f"  Units: {unit_data.n_units:,}")
print(f"  Times: {unit_data.n_times}")
print(f"  Treatment groups: {len(unit_data.treatment_groups)}")
print(f"  (g,t) pairs: {gt_info.n_gt}")
print(f"  Unique base periods: {len(gt_info.unique_base_periods)}")
print(f"  Covariates: {covariate_info.n_covariates}")

## Cell 4: Cross-Fitting

**This is the computationally intensive step.** Each fold trains the model and computes out-of-fold predictions for all (g,t) pairs in a single forward pass.

In [ ]:
print("=" * 60)
print("CROSS-FITTING")
print("=" * 60)

timer = Timer()

cf_results = run_cross_fitting(
    unit_data,
    gt_info,
    covariate_info,
    config
)

training_time = timer.elapsed()
print(f"\nCross-fitting complete in {training_time:.1f}s ({training_time/60:.1f}m)")

# Validate
validate_cross_fitting(cf_results)

## Cell 5: ATT Estimation

In [ ]:
print("=" * 60)
print("ATT ESTIMATION")
print("=" * 60)

timer = Timer()

att_results = compute_all_att(cf_results, config)
print_att_summary(att_results)

print(f"\nATT estimation complete in {timer.elapsed():.1f}s")

## Cell 6: Inference (Bootstrap)

In [ ]:
print("=" * 60)
print("INFERENCE")
print("=" * 60)

timer = Timer()

# Add bootstrap inference
att_results = add_bootstrap_inference(att_results, config)

# Test parallel trends
pt_test = test_parallel_trends(att_results, config)

# Simple ATT
simple_att = compute_simple_att(att_results, config)

print(f"\nInference complete in {timer.elapsed():.1f}s")

print("\n--- Parallel Trends Test ---")
print(f"Mean pre-treatment ATT: {pt_test['mean_att_pre']:.4f} (SE: {pt_test['se_mean_pre']:.4f})")
print(f"p-value: {pt_test['p_value']:.4f}")
print(f"Reject at 5%: {'YES' if pt_test['reject'] else 'NO'}")

print("\n--- Simple ATT ---")
print(f"ATT: {simple_att['att']:.4f} (SE: {simple_att['se']:.4f})")
print(f"95% CI: [{simple_att['ci_lower']:.4f}, {simple_att['ci_upper']:.4f}]")
print(f"p-value: {simple_att['p_value']:.4f}")

## Cell 7: Aggregation

In [ ]:
print("=" * 60)
print("AGGREGATION")
print("=" * 60)

agg_results = aggregate_all(att_results, config)
print_aggregation_summary(agg_results)

# Display event study table
print("\n--- Event Study Estimates ---")
es_df = agg_results['event_study']['event_study']
display_cols = ['event_time', 'att', 'se', 'ci_lower', 'ci_upper']
if 'uniform_lower' in es_df.columns:
    display_cols += ['uniform_lower', 'uniform_upper']
print(es_df[display_cols].to_string(index=False))

## Cell 8: Visualization

In [ ]:
print("=" * 60)
print("VISUALIZATION")
print("=" * 60)

output_dir = os.path.join(project_root, "outputs/figures")
os.makedirs(output_dir, exist_ok=True)

# Event study plot
fig = plot_event_study(
    agg_results['event_study'],
    title="Effect of CDFI Lending on Startup Formation Rate",
    subtitle="Callaway & Sant'Anna (2021) DiD with Neural Network Nuisance Estimation",
    show_uniform_bands=True,
    show_pointwise_ci=True
)
plt.show()

# Save
save_event_study(fig, "event_study", output_dir)
print(f"\nFigures saved to: {output_dir}")

## Cell 9: Save Results

In [ ]:
import pickle

results = {
    'config': config,
    'unit_data': unit_data,
    'gt_info': gt_info,
    'covariate_info': covariate_info,
    'cf_results': cf_results,
    'att_results': att_results,
    'agg_results': agg_results,
    'parallel_trends_test': pt_test,
    'simple_att': simple_att,
    'training_time': training_time
}

output_path = os.path.join(project_root, "outputs/estimation_results.pkl")
with open(output_path, 'wb') as f:
    pickle.dump(results, f)

print(f"Results saved to: {output_path}")

## Cell 10: Summary

In [ ]:
print("\n" + "=" * 60)
print("ESTIMATION COMPLETE")
print("=" * 60)

print("\nKEY RESULTS:\n")

print("1. Simple ATT (weighted average post-treatment):")
print(f"   ATT = {simple_att['att']:.4f}, SE = {simple_att['se']:.4f}")
print(f"   95% CI: [{simple_att['ci_lower']:.4f}, {simple_att['ci_upper']:.4f}]")
print(f"   p-value: {simple_att['p_value']:.4f}\n")

print("2. Parallel Trends:")
print(f"   Pre-treatment ATT = {pt_test['mean_att_pre']:.4f} (should be ~0)")
print(f"   p-value = {pt_test['p_value']:.4f} (want > 0.05)\n")

print("3. Event Study:")
es = agg_results['event_study']['event_study']
print(f"   Event times: {es['event_time'].min()} to {es['event_time'].max()}")
print(f"   Pre-treatment mean: {es[es['event_time'] < 0]['att'].mean():.4f}")
print(f"   Post-treatment mean: {es[es['event_time'] >= 0]['att'].mean():.4f}")

print(f"\nTraining time: {training_time/60:.1f} minutes")
print(f"Figures: {output_dir}")
print(f"Results: {output_path}")